# Part 2 · The agent, streamed as Server-Sent Events

A brief takes tens of seconds. Instead of one JSON answer at the end, the service streams the run: each step of
the agent becomes an **SSE event** the UI shows as it happens. This part reads the three pieces that make that work:

| File | Role |
| --- | --- |
| [`agent/tools.py`](../app/agent/tools.py), [`agent/graph.py`](../app/agent/graph.py) | the Research Brief Agent (Deep Agent, web search, `save_brief` that waits for approval) |
| [`services/streaming.py`](../app/services/streaming.py) | LangGraph stream chunks → named events (`to_events`), one whole run as SSE (`run_events`) |
| [`routers/runs.py`](../app/routers/runs.py) | `POST /threads/{thread_id}/runs` returns that stream with `EventSourceResponse` |

Docs: [FastAPI Server-Sent Events](https://fastapi.tiangolo.com/tutorial/server-sent-events/) ·
[LangGraph streaming](https://docs.langchain.com/oss/python/langgraph/streaming) ·
[MDN: Using server-sent events](https://developer.mozilla.org/en-US/docs/Web/API/Server-sent_events/Using_server-sent_events)

In [1]:
import logging

from dotenv import load_dotenv

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
for noisy in ("httpx", "httpx2"):  # HTTP clients log every request at INFO
    logging.getLogger(noisy).setLevel(logging.WARNING)
load_dotenv()

from utils import check_server, client, short, show, source  # notebooks/utils.py

check_server()

server: ok


## 1. The agent

`build_agent` is called once by the lifespan. `interrupt_on` pauses the run before `save_brief`, and the tools read
the user from the run **context** (filled from the token), never from the model's arguments. `internet_search` also
reports progress with `get_stream_writer()`: that becomes the `progress` event.

In [2]:
from app.agent import graph, tools

source(graph.build_agent)

def build_agent(model, checkpointer=None):
    """Built once at startup (lifespan); the checkpointer keeps every thread so interrupts can be resumed."""
    return create_deep_agent(
        model=model,
        tools=[internet_search, save_brief],
        system_prompt=PROMPT,
        context_schema=Context,
        interrupt_on={"save_brief": {"allowed_decisions": ["approve", "reject"]}},  # HumanInTheLoopMiddleware
        middleware=[ToolCallLimitMiddleware(tool_name="internet_search", run_limit=3, exit_behavior="continue")],
        checkpointer=checkpointer,
        name="research_brief",
    )

In [3]:
source(tools.internet_search)

@tool(parse_docstring=True)
def internet_search(query: str) -> list[dict]:
    """Search the web and return title, url and a short snippet for the 3 best results.

    Args:
        query: What to search for, in a few keywords.
    """
    global _tavily
    started, status = time.monotonic(), "error"
    get_stream_writer()({"stage": "searching", "query": query})  # stream_mode="custom": progress for the UI
    try:
        _tavily = _tavily or TavilyClient(api_key=get_settings().tavily_api_key.get_secret_value())
        results = _tavily.search(query, max_results=3)["results"]  # 3 results keep each call small and cheap
        status = "ok"
        return [{"title": r["title"], "url": r["url"], "content": r["content"][:500]} for r in results]
    except Exception as exc:  # quota, network, bad key: tell the model instead of crashing the whole run
        return [{"error": f"search failed: {exc}"}]
    finally:
        log_call("internet_search", repr(query), status, started)

## 2. From LangGraph chunks to events: `to_events`

`agent.astream(..., stream_mode=["updates", "messages", "custom"])` yields `(mode, chunk)` pairs. `to_events` maps
each one to zero or more `(event, data)` pairs. The `values` mode (whole state after each step) is not used: it
repeats every message each time and would expose internal state to the browser.

In [4]:
from app.services.streaming import to_events

source(to_events)

def to_events(mode: str, chunk) -> list[tuple[str, dict]]:
    """One LangGraph stream chunk -> zero or more (event name, data) pairs."""
    if mode == "messages":  # (message chunk, metadata): only text tokens of the agent's own model node
        token, meta = chunk
        ok = isinstance(token, AIMessageChunk) and token.text and meta.get("langgraph_node") == "model"
        return [("token", {"text": token.text})] if ok else []
    if mode == "custom":  # whatever a tool wrote with get_stream_writer()
        return [("progress", chunk)]
    events = []  # updates: {node name: what the node returned}
    for node, update in chunk.items():
        if node == "__interrupt__":
            events.append(("interrupt", {"id": update[0].id, **update[0].value}))
            continue
        if not isinstance(update, dict):
            continue
        if node in ("model", "tools"):
            events.append(("step", {"node": node}))
        for m in update.get("messages", []):  # model, tools, or a middleware hook such as a call limit
            for c in getattr(m, "tool_calls", []) or []:
                events.append(("tool_call", {"id": c["id"], "name": c["name"], "args": c["args"]}))
            if isinstance(m, ToolMessage):
                events.append(("tool_result", {"id": m.tool_call_id, "name": m.name, "status": m.status,
                                               "preview": short(m.text)}))
    return events

Run the agent **in this notebook** (in-memory checkpointer) and print, for each chunk, its mode, its raw shape and
the events it becomes. Tokens are counted, not printed: this question ends at the approval step, and the model writes reply text only after the decision (Part 3), so the count is 0 here. Middleware hooks such as `HumanInTheLoopMiddleware.after_model` give no event unless they add a message (the call limit adds an error `ToolMessage`, hence one `tool_result`).

In [5]:
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import InMemorySaver

from app.agent.tools import Context
from app.core.config import get_settings

agent = graph.build_agent(init_chat_model(get_settings().model), InMemorySaver())
config = {"configurable": {"thread_id": "notebook-1"}}
payload = {"messages": [{"role": "user", "content": "Brief on how loud heat pumps are for neighbours."}]}

tokens = 0
async for mode, chunk in agent.astream(payload, config, context=Context(user_id="alice"),
                                       stream_mode=["updates", "messages", "custom"]):
    events = to_events(mode, chunk)
    if mode == "messages":
        tokens += len(events)
        continue
    raw = list(chunk) if mode == "updates" else chunk
    print(f"{mode:<8} {short(raw, 60):<62} -> {[name for name, _ in events]}")
print("token events:", tokens)

updates  ['PatchToolCallsMiddleware.before_agent']                      -> []
updates  ['model']                                                      -> ['step', 'tool_call']
updates  ['HumanInTheLoopMiddleware.after_model']                       -> []
updates  ['ToolCallLimitMiddleware[internet_search].after_model']       -> []
custom   {'stage': 'searching', 'query': 'air source heat pump outdoo…  -> ['progress']


INFO | internet_search('air source heat pump outdoor unit noise neighbour dB distance governm…) -> ok 1987 ms


updates  ['tools']                                                      -> ['step', 'tool_result']
updates  ['model']                                                      -> ['step', 'tool_call']
updates  ['HumanInTheLoopMiddleware.after_model']                       -> []
updates  ['ToolCallLimitMiddleware[internet_search].after_model']       -> []
custom   {'stage': 'searching', 'query': 'site:gov.uk heat pump noise…  -> ['progress']


INFO | internet_search('site:gov.uk heat pump noise 42 dB neighbour sound pressure level guid…) -> ok 6262 ms


updates  ['tools']                                                      -> ['step', 'tool_result']
updates  ['model']                                                      -> ['step', 'tool_call']
updates  ['HumanInTheLoopMiddleware.after_model']                       -> []
updates  ['ToolCallLimitMiddleware[internet_search].after_model']       -> []
custom   {'stage': 'searching', 'query': 'heat pump outdoor unit soun…  -> ['progress']


INFO | internet_search('heat pump outdoor unit sound power level typical dB manufacturer nois…) -> ok 2625 ms


updates  ['tools']                                                      -> ['step', 'tool_result']
updates  ['model']                                                      -> ['step', 'tool_call']
updates  ['HumanInTheLoopMiddleware.after_model']                       -> []
updates  ['ToolCallLimitMiddleware[internet_search].after_model']       -> ['tool_result']
updates  ['model']                                                      -> ['step', 'tool_call']
updates  ['__interrupt__']                                              -> ['interrupt']
token events: 0


## 3. A whole run: `run_events`

`run_events` wraps the loop: it sends `metadata` first, the mapped events, then `done` (or `error`); it gives every
event an `id`, enforces one run per thread and a time limit for the whole run. It takes plain objects (agent, the
registry of running threads, user, thread), not the HTTP request, so it can be called here directly.

In [6]:
from app.services.streaming import run_events

source(run_events)

async def run_events(agent, running: dict[str, str], user_id: str, thread_id: str, payload,
                     timeout_s: int = 180) -> AsyncIterator[ServerSentEvent]:
    """Stream one run. `running` maps thread_id -> run_id so a thread never has two runs at once."""
    run_id, seq, started = uuid.uuid4().hex[:8], 0, time.monotonic()
    config = {"configurable": {"thread_id": thread_id}, "recursion_limit": 40}

    def event(name: str, data: dict) -> ServerSentEvent:
        nonlocal seq
        seq += 1
        return ServerSentEvent(event=name, data=data, id=f"{run_id}-{seq}")

    if thread_id in running:  # a second request passed the 409 check at the same moment
        yield event("error", problem(409, "A run is already streaming on this thread", f"/threads/{thread_id}"))
        return
    running[thread_id] = run_id  # claimed with no await since the check above
    logger.info("run %s started: user=%s thread=%s", run_id, user_id, thread_id)
    stream = agent.astream(payload, config, context=Context(user_id=user_id),
                           stream_mode=["updates", "messages", "custom"])
    try:
        yield event("metadata", {"run_id": run_id, "thread_id": thread_id})
        while True:
            try:  # the deadline covers the whole run, not each chunk
                mode, chunk = await asyncio.wait_for(anext(stream), timeout_s - (time.monotonic() - started))
            except StopAsyncIteration:
                break
            for name, data in to_events(mode, chunk):
                yield event(name, data)
        state = await agent.aget_state(config)
        last = state.values["messages"][-1]
        done = {"status": "interrupted" if state.interrupts else "completed",
                "answer": last.text if isinstance(last, AIMessage) and not state.interrupts else None,
                "seconds": round(time.monotonic() - started, 1)}
        logger.info("run %s %s in %.0f s", run_id, done["status"], done["seconds"])
        yield event("done", done)
    except asyncio.CancelledError:  # the client closed the connection: stop the agent, no new model/tool calls
        logger.warning("run %s cancelled: client disconnected", run_id)
        raise
    except TimeoutError:
        logger.warning("run %s timed out after %d s", run_id, timeout_s)
        yield event("error", problem(504, f"The run took longer than {timeout_s} s.", f"/threads/{thread_id}"))
    except Exception as exc:  # status 200 was sent with the first byte: report the failure as an event
        logger.error("run %s failed: %r", run_id, exc)
        yield event("error", problem(500, f"The run stopped ({type(exc).__name__}); see the server log, run {run_id}.",
                                     f"/threads/{thread_id}"))
    finally:
        await stream.aclose()
        if running.get(thread_id) == run_id:
            del running[thread_id]

FastAPI turns each `ServerSentEvent` into text lines. `format_sse_event` is the function it uses; the cell runs a
new question through `run_events` and prints the first events **exactly as they go on the wire**, then stops reading.

In [7]:
import json

from fastapi.sse import format_sse_event

payload = {"messages": [{"role": "user", "content": "Brief on induction cooktops vs gas."}]}
stream = run_events(agent, {}, "alice", "notebook-2", payload)
for _ in range(4):
    e = await anext(stream)
    print(format_sse_event(data_str=json.dumps(e.data), event=e.event, id=e.id).decode(), end="")
await stream.aclose()  # stop the run: no more model or tool calls

INFO | run 6234a28c started: user=alice thread=notebook-2


event: metadata
data: {"run_id": "6234a28c", "thread_id": "notebook-2"}
id: 6234a28c-1

event: step
data: {"node": "model"}
id: 6234a28c-2

event: tool_call
data: {"id": "call_2YSEYsGBjXanhHkusTFDajM9", "name": "internet_search", "args": {"query": "induction cooktop vs gas efficiency energy use safety indoor air pollution research"}}
id: 6234a28c-3

event: progress
data: {"stage": "searching", "query": "induction cooktop vs gas efficiency energy use safety indoor air pollution research"}
id: 6234a28c-4



## 4. The route and the real HTTP stream

`create_run` declares `response_class=EventSourceResponse` and yields the events of `run_events`. Its parameters are
the checks of Part 3 (`IdleThread`, `CurrentUser`): they run **before** the first byte is sent.

In [8]:
from app.routers import runs

source(runs.create_run)

@router.post("/runs", response_class=EventSourceResponse)
async def create_run(body: RunRequest, thread: IdleThread, user: CurrentUser, agent: Agent, store: Store,
                     running: Running) -> AsyncIterable[ServerSentEvent]:
    """Send a message to the agent on this thread; the response streams the run as Server-Sent Events."""
    if thread.title == "New brief":  # the first message names the thread in the list
        await threads.set_title(store, user.user_id, thread, short(body.message, 60))
    payload = {"messages": [{"role": "user", "content": body.message}]}
    async for event in run_events(agent, running, user.user_id, thread.thread_id, payload,
                                  get_settings().run_timeout_s):
        yield event

Now the running service: create a thread as Alice, send a message, print the response headers and every event in
the order it arrives. The run pauses before `save_brief`: the stream sends `interrupt`, then `done` with status
`interrupted`, and closes (Part 3 resumes it).

In [9]:
from utils import sse_events

alice = client("alice")
thread_id = alice.post("/threads").json()["thread_id"]
events = []
with alice.stream("POST", f"/threads/{thread_id}/runs",
                  json={"message": "Brief on how well air-source heat pumps work below 0°C."}) as r:
    print("status:", r.status_code, "| content-type:", r.headers["content-type"], "| cache-control:",
          r.headers["cache-control"], "\n")
    for e in sse_events(r):
        events.append(e)
        show(e["event"], e["data"])

status: 200 | content-type: text/event-stream; charset=utf-8 | cache-control: no-cache 

[metadata] run 5f004a84 on thread 05dfd782d1cb
[step] model
[tool_call] internet_search({"query": "air source heat pump performance below 0°C cold weather efficiency COP field st…)
[progress] searching: air source heat pump performance below 0°C cold weather efficiency COP field study government
[step] tools
[tool_result] internet_search -> success: [{"title": "Weather in global", "url": "https://www.weatherapi.com/", "content":…
[step] model
[tool_call] internet_search({"query": "site:energy.gov air source heat pump cold weather below 0 F performance efficie…)
[progress] searching: site:energy.gov air source heat pump cold weather below 0 F performance efficiency COP cold climate
[step] tools
[tool_result] internet_search -> success: [{"title": "Field Performance of Inverter-Driven Heat Pumps in Cold ...", "url":…
[step] model
[tool_call] internet_search({"query": "air source heat pumps cold weath

In [10]:
from collections import Counter

print("events:", dict(Counter(e["event"] for e in events)))
action = next(e["data"] for e in events if e["event"] == "interrupt")["action_requests"][0]
print("waiting for approval:", action["name"], "| title:", action["args"]["title"])
print("thread status:", alice.get(f"/threads/{thread_id}").json()["status"])

events: {'metadata': 1, 'step': 7, 'tool_call': 4, 'progress': 3, 'tool_result': 3, 'interrupt': 1, 'done': 1}
waiting for approval: save_brief | title: Air-source heat pumps below 0°C
thread status: interrupted
